# 05 Firm characteristics

Builds the monthly panel, the excess returns and the twenty-two characteristics of Drobetz and Otto (2021). Row (i, t) holds the characteristics known at the end of month t and, as target, the excess return of month t + 1; in the definitions, month −1 is the current row. Market data enter without delay, accounting data four months after the fiscal year end. Sections 3.3, 3.4.4 and 3.4.5.

## Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Thesis/data"

monthly = pd.read_parquet(os.path.join(DATA_DIR, "dedup_monthly.parquet"))
weekly  = pd.read_parquet(os.path.join(DATA_DIR, "dedup_weekly.parquet"))
annual  = pd.read_parquet(os.path.join(DATA_DIR, "raw_annual.parquet"))
fye     = pd.read_parquet(os.path.join(DATA_DIR, "fiscal_year_end.parquet"))
rf_raw  = pd.read_parquet(os.path.join(DATA_DIR, "raw_riskfree.parquet"))
idx_raw = pd.read_parquet(os.path.join(DATA_DIR, "raw_index.parquet"))
master  = pd.read_csv(os.path.join(DATA_DIR, "master_universe.csv"), dtype=str)

for nm, d in [("monthly", monthly), ("weekly", weekly), ("annual", annual)]:
    print(f"{nm:8s} {len(d):>12,} rows | {d['symbol'].nunique():>6,} securities")
print(f"\nindices: {sorted(idx_raw['symbol'].unique())}")
print(f"risk-free: {len(rf_raw)} observations")

## Functions

In [ ]:
import numpy as np
import pandas as pd


def build_skeleton(monthly_long):
    """Wide panel on a continuous monthly calendar, per security."""
    w = monthly_long.pivot_table(index=["symbol", "date"], columns="datatype",
                                 values="value", aggfunc="first")
    w.columns.name = None
    rng = w.reset_index().groupby("symbol")["date"].agg(["min", "max"])
    idx = []
    for sym, (d0, d1) in rng.iterrows():
        dates = pd.date_range(d0, d1, freq="MS")
        idx.append(pd.MultiIndex.from_product([[sym], dates], names=["symbol", "date"]))
    full = idx[0].append(idx[1:]) if len(idx) > 1 else idx[0]
    return w.reindex(full).sort_index()


def add_returns(p, rf, apply_reversal_filter=True):
    """Simple and excess returns from the total return index, with the reversal filter of Ince and Porter (2006)."""
    p = p.copy()
    ri = p["RI"].where(p["RI"] > 0)
    p["ret"] = ri.groupby(level="symbol").pct_change(fill_method=None)
    p["ret"] = p["ret"].replace([np.inf, -np.inf], np.nan)

    if apply_reversal_filter:
        r = p["ret"]
        r_prev = r.groupby(level="symbol").shift(1)
        pair = (1 + r) * (1 + r_prev) - 1
        bad = ((r > 3.0) | (r_prev > 3.0)) & (pair < 0.5)
        bad = bad.fillna(False)
        n = int(bad.sum())
        p.loc[bad, "ret"] = np.nan
        # the paired observation is discarded as well
        bad_next = bad.groupby(level="symbol").shift(-1, fill_value=False).astype(bool)
        p.loc[bad_next, "ret"] = np.nan
        p["_rev_filtered"] = bad | bad_next

    p = p.join(rf.rename("rf"), on="date")
    p["rf"] = p["rf"] / 100.0 / 12.0          # annual percent -> monthly decimal
    p["retx"] = p["ret"] - p["rf"]
    return p


def _cum_window(s, start_lag, n):
    """Cumulative return over the n months ending `start_lag` months back."""
    lr = np.log1p(s.clip(lower=-0.99))
    roll = (lr.groupby(level="symbol")
              .apply(lambda x: x.shift(start_lag).rolling(n, min_periods=n).sum())
              .droplevel(0))
    return np.expm1(roll.reindex(s.index))


def add_market_characteristics(p):
    """Characteristics computable from monthly market and I/B/E/S data."""
    p = p.copy()
    gb = lambda c: p.groupby(level="symbol")[c]

    # size
    p["size"] = np.log(p["MV"].where(p["MV"] > 0))

    # momentum and reversal
    p["ret_1"] = p["retx"]
    p["ret_2_12"] = _cum_window(p["retx"], 1, 11)
    p["ret_12_36"] = _cum_window(p["retx"], 11, 25)

    # Share issuance, adjusted for capital events.
    # Defined as growth "from month -36 to month -1". Indexing months relative
    # to the month whose return is being predicted, month -1 is the current
    # panel row and month -36 is thirty-five rows earlier.
    adj = (p["NOSH"] / p["AF"]).replace([np.inf, -np.inf], np.nan)
    adj = adj.where(adj > 0)
    p["_adj_sh"] = adj
    la = np.log(adj)
    p["issues_1_12"] = la - gb("_adj_sh").shift(11).pipe(np.log)
    p["issues_1_36"] = la - gb("_adj_sh").shift(35).pipe(np.log)

    # turnover: shares traded over shares outstanding, averaged over 12 months
    to = (p["VO"] / p["NOSH"]).replace([np.inf, -np.inf], np.nan)
    p["_to"] = to
    p["turnover"] = (p.groupby(level="symbol")["_to"]
                       .apply(lambda x: x.rolling(12, min_periods=6).mean())
                       .droplevel(0).reindex(p.index))

    # Dividend yield. The Datastream dividend per share is already a trailing
    # twelve-month figure reported monthly: verified against the DY datatype,
    # DPS/P reproduces it exactly, whereas summing DPS over a twelve-month
    # window overstates it by a factor of about twelve.
    p["dy"] = p["DPS"] / p["P"].where(p["P"] > 0)

    # forecast dispersion
    sd = p["EPS1SD"].where(p["EPS1SD"] > 0)
    mn = p["EPS1MN"].abs().replace(0, np.nan)
    p["_fc"] = np.log(sd) - np.log(mn)
    p["fcdispersion"] = (p.groupby(level="symbol")["_fc"]
                           .apply(lambda x: x.rolling(12, min_periods=3).mean())
                           .droplevel(0).reindex(p.index))

    return p.drop(columns=[c for c in p.columns if c.startswith("_")])


def add_target(p):
    """Dependent variable: excess return realised in the following month."""
    p = p.copy()
    p["target"] = p.groupby(level="symbol")["retx"].shift(-1)
    return p

def apply_price_screen(p, threshold=0.25):
    """Price screen of Ince and Porter (2006): returns are set to missing when the previous month's price is below the threshold."""
    p = p.copy()
    prev_price = p.groupby(level="symbol")["P"].shift(1)
    below = (prev_price < threshold) & prev_price.notna()
    p.loc[below, ["ret", "retx"]] = np.nan
    p["_price_screened"] = below
    return p


In [ ]:
import numpy as np
import pandas as pd

# Benchmark

def build_composite_index(index_long, monthly_panel, master, codes):
    """Euro-area benchmark: national total market indices weighted by the sample market value of each country in the previous month."""
    idx = (index_long[index_long["symbol"].isin(codes.values())]
           .pivot_table(index="date", columns="symbol", values="value", aggfunc="first")
           .sort_index())
    ret = idx.pct_change(fill_method=None)

    mv = monthly_panel.reset_index()[["symbol", "date", "MV"]].dropna()
    mv = mv.merge(master[["Symbol", "country"]], left_on="symbol", right_on="Symbol")
    agg = mv.groupby(["date", "country"])["MV"].sum().unstack("country")
    agg = agg.shift(1)                                   # weights known ex ante
    w = agg.div(agg.sum(axis=1), axis=0)

    w = w.rename(columns={c: codes[c] for c in codes if c in w.columns})
    w = w.reindex(ret.index, method="ffill")

    cols = [c for c in ret.columns if c in w.columns]
    wn = w[cols].div(w[cols].sum(axis=1), axis=0)
    mkt = (ret[cols] * wn).sum(axis=1, min_count=1)
    mkt.name = "mkt_ret"
    return mkt, wn


# Weekly beta and volatility

def first_panel_month(dates):
    """
    First panel row in which information dated `dates` is known. Rows are dated on
    the first day of each month and hold the information available on that day;
    information dated later in month m can be used only from row m + 1.
    """
    d = pd.to_datetime(pd.Series(dates)).reset_index(drop=True)
    m = (d - pd.Timedelta(days=1)).values.astype("datetime64[M]") + np.timedelta64(1, "M")
    return pd.to_datetime(m).astype("datetime64[ns]")


def beta_and_vola(weekly_long, mkt_ret, beta_weeks=156, vola_weeks=52,
                  min_beta=104, min_vola=35):
    """
    Market beta and return volatility from weekly returns, over the thirty-six and
    twelve months to t-1 (Drobetz and Otto, 2021). Each estimate is assigned to the
    first panel row in which all its weekly returns are known, so that no week of the
    forecast month enters it; covariance and market variance use the same weeks.
    """
    ri = weekly_long[(weekly_long["datatype"] == "RI") & (weekly_long["value"] > 0)]
    w = (ri.pivot_table(index="date", columns="symbol", values="value", aggfunc="first")
           .sort_index())
    r = w.pct_change(fill_method=None)
    r = r.replace([np.inf, -np.inf], np.nan)

    m = mkt_ret.reindex(r.index)
    valid = r.notna() & m.notna().to_numpy()[:, None]
    r = r.where(valid)
    M = pd.DataFrame(np.where(valid, m.to_numpy()[:, None], np.nan),
                     index=r.index, columns=r.columns)

    roll = lambda x: x.rolling(beta_weeks, min_periods=min_beta).mean()
    rm, mm = roll(r), roll(M)
    beta = (roll(r * M) - rm * mm) / (roll(M * M) - mm * mm)
    vola = r.rolling(vola_weeks, min_periods=min_vola).std()

    def to_monthly(df, name):
        s = df.stack(future_stack=True).rename(name).reset_index()
        s.columns = ["date", "symbol", name]
        s["month"] = first_panel_month(s["date"]).values
        s = s.sort_values("date").groupby(["symbol", "month"]).last()
        return s[name]

    out = pd.concat([to_monthly(beta, "beta"), to_monthly(vola, "vola")], axis=1)
    out.index.names = ["symbol", "date"]
    return out

# Accounting data

def accounting_available(annual_long, fy_long, lag_months=4):
    """
    Wide annual accounting data with the date from which each observation may
    be used, being the fiscal period end plus the reporting lag.
    """
    a = annual_long.pivot_table(index=["symbol", "date"], columns="datatype",
                                values="value", aggfunc="first")
    a.columns.name = None
    a = a.reset_index().rename(columns={"date": "fy"})

    fy = fy_long[["symbol", "date", "value_date"]].rename(
        columns={"date": "fy", "value_date": "fye"})
    a = a.merge(fy, on=["symbol", "fy"], how="left")

    # fall back on a December year-end where the reported date is missing
    a["fye"] = a["fye"].fillna(pd.to_datetime(a["fy"].astype(str) + "-12-31"))
    # Worldscope reports in thousands of currency units, Datastream market value in millions.
    for c in [c for c in a.columns if c.startswith("WC")]:
        a[c] = a[c] / 1000.0

    a["fye"] = pd.to_datetime(a["fye"])
    # known `lag_months` after the fiscal year-end: first used in the row that begins on
    # or after that date (December year-end -> row of 1 May, forecasting May returns)
    a["available"] = first_panel_month(a["fye"] + pd.DateOffset(months=lag_months)).values
    return a.sort_values(["symbol", "available"])


def add_accounting_characteristics(panel, acc):
    """
    Merge accounting data onto the monthly panel as of the date each fiscal
    year became available, then form the accounting-based characteristics.
    """
    acc = acc.copy().sort_values(["symbol", "fy"])
    g = acc.groupby("symbol")

    # Prior-year levels, for growth rates and averages. Where a fiscal year is
    # missing from the record, the preceding available year is not a valid
    # comparison and the lagged level is discarded, so that a two-year change
    # is not treated as an annual one.
    consecutive = (acc["fy"] - g["fy"].shift(1)) == 1
    for c in ["WC02999", "WC02201", "WC02003", "WC03101", "WC03051", "WC03063"]:
        acc[f"{c}_lag"] = g[c].shift(1).where(consecutive)

    avg_ta = (acc["WC02999"] + acc["WC02999_lag"]) / 2
    avg_ta = avg_ta.where(avg_ta > 0)

    acc["operatingprofitability"] = acc["WC18191"] / avg_ta
    acc["grossprofitability"] = (acc["WC01001"] - acc["WC01051"]) / avg_ta
    acc["roa"] = acc["WC01551"] / avg_ta
    acc["totalassetgrowth"] = np.log(acc["WC02999"].where(acc["WC02999"] > 0)) - \
                              np.log(acc["WC02999_lag"].where(acc["WC02999_lag"] > 0))
    acc["investment"] = acc["WC04601"] / acc["WC01001"].where(acc["WC01001"] > 0)

    # Sloan (1996) working capital accruals
    d_ca = acc["WC02201"] - acc["WC02201_lag"]
    d_cash = acc["WC02003"] - acc["WC02003_lag"]
    d_cl = acc["WC03101"] - acc["WC03101_lag"]
    d_std = acc["WC03051"] - acc["WC03051_lag"]
    d_tp = acc["WC03063"] - acc["WC03063_lag"]
    acc["accrualschange"] = ((d_ca - d_cash) - (d_cl - d_std - d_tp) - acc["WC01151"]) / avg_ta

    cols = ["symbol", "available", "fy", "fye",
            "WC03501", "WC01751", "WC01001", "WC01151", "WC03051", "WC03251",
            "operatingprofitability", "grossprofitability", "roa",
            "totalassetgrowth", "investment", "accrualschange"]
    acc = acc[cols].dropna(subset=["available"])

    p = panel.reset_index().sort_values("date")
    p["date"] = pd.to_datetime(p["date"]).astype("datetime64[ns]")
    merged = pd.merge_asof(p, acc.sort_values("available"),
                           left_on="date", right_on="available",
                           by="symbol", direction="backward")
    merged = merged.set_index(["symbol", "date"]).sort_index()

    mv = merged["MV"].where(merged["MV"] > 0)
    merged["bm"] = np.log(merged["WC03501"].where(merged["WC03501"] > 0)) - np.log(mv)
    merged["earningstoprice"] = merged["WC01751"] / mv
    merged["salestoprice"] = merged["WC01001"] / mv
    merged["cftoprice"] = (merged["WC01751"] + merged["WC01151"]) / mv
    debt = merged[["WC03051", "WC03251"]].sum(axis=1, min_count=1)
    merged["debttoprice"] = debt / mv
    return merged


## Panel, returns and data screens

Each security is placed on a continuous monthly calendar, and excess returns are computed from the total return index over the monthly three-month EURIBOR. Three screens of Ince and Porter (2006) are applied: non-positive return index, reversal filter, and a price screen at €0.25.

In [ ]:
rf = rf_raw.set_index("date")["value"]

panel = build_skeleton(panel_input := monthly)
panel = add_returns(panel, rf, apply_reversal_filter=True)

print(f"Panel: {len(panel):,} security-months, "
      f"{panel.index.get_level_values('symbol').nunique():,} securities")
if "_rev_filtered" in panel:
    print(f"Removed by reversal filter: {int(panel['_rev_filtered'].sum()):,}")

# price screen: effect of alternative thresholds
prev_price = panel.groupby(level="symbol")["P"].shift(1)
print("\nEffect of the price screen on excess returns:")
for thr in [0.00, 0.10, 0.25, 1.00]:
    r = panel["retx"].where((prev_price >= thr) | prev_price.isna())
    print(f"  threshold EUR {thr:4.2f}: n={r.notna().sum():>9,}  "
          f"std={r.std():>8.3f}  max={r.max():>9.1f}")

PRICE_SCREEN = 0.25
panel = apply_price_screen(panel, threshold=PRICE_SCREEN)
print(f"\nApplied at EUR {PRICE_SCREEN:.2f}: "
      f"{int(panel['_price_screened'].sum()):,} observations set to missing")
print(f"\nMonthly excess returns: {panel['retx'].notna().sum():,}")
print(panel["retx"].describe(percentiles=[.01, .5, .99]).round(4).to_string())

## Market benchmark

The benchmark is not specified by Drobetz and Otto (2021). A euro-area composite of the four national total market indices is used, weighted by the sample market value of each country at the end of the previous month.

In [ ]:
INDEX_CODES = {"DE": "TOTMKBD", "FR": "TOTMKFR", "IT": "TOTMKIT", "ES": "TOTMKES"}

mkt, weights = build_composite_index(idx_raw, panel, master, INDEX_CODES)

print(f"Composite benchmark: {mkt.notna().sum():,} weekly returns")
print(f"  {mkt.index.min():%Y-%m} to {mkt.index.max():%Y-%m}")
print(f"  mean {mkt.mean()*100:.3f}% per week, std {mkt.std()*100:.2f}%")
print("\nAverage country weights:")
print((weights.mean() * 100).round(1).astype(str).add("%").to_string())

## Characteristics

### Market-based

Shares are split-adjusted as `NOSH / AF`: with the adjustment factor, MV = (P/AF) × NOSH/1000 holds within 5% for 94.5% of observations, against 79.5% without it. `DPS` is already a trailing twelve-month figure, so `dy` is `DPS / P`.

In [ ]:
panel = add_market_characteristics(panel)
panel = add_target(panel)

MARKET = ["size", "ret_1", "ret_2_12", "ret_12_36",
          "issues_1_12", "issues_1_36", "turnover", "dy", "fcdispersion"]
for ch in MARKET:
    print(f"  {ch:14s} {panel[ch].notna().sum():>10,}  ({100*panel[ch].notna().mean():>5.1f}%)")

### Beta and volatility

Rolling covariance on weekly returns over 156 and 52 weeks, with at least two thirds of each window required.

In [ ]:
bv = beta_and_vola(weekly, mkt, beta_weeks=156, vola_weeks=52,
                   min_beta=104, min_vola=35)
panel = panel.join(bv, how="left")

print(f"  beta  {panel['beta'].notna().sum():>10,}  ({100*panel['beta'].notna().mean():>5.1f}%)")
print(f"  vola  {panel['vola'].notna().sum():>10,}  ({100*panel['vola'].notna().mean():>5.1f}%)")
print()
print(panel[["beta", "vola"]].describe(percentiles=[.01, .5, .99]).round(3).to_string())

Volatility is annualised (× √52), consistent with the mean of 0.32 reported by Drobetz and Otto. The rank transformation makes the choice irrelevant for the models.

In [ ]:
ANNUALISE_VOLA = True

if ANNUALISE_VOLA:
    panel["vola"] = panel["vola"] * np.sqrt(52)
    print(f"Annualised. Mean {panel['vola'].mean():.3f}, "
          f"median {panel['vola'].median():.3f} (Drobetz and Otto report 0.32)")
else:
    print(f"Weekly units. Mean {panel['vola'].mean():.4f}")

### Accounting-based

Each fiscal year enters four months after its reported period end (WC05350); 11.2% of firm-years do not end in December. Accounting items are divided by 1,000, since Worldscope reports in thousands and Datastream market value in millions. The four cases counted as 3 have a fiscal year end dated on the first day of a month and still enter four months later.

In [ ]:
acc = accounting_available(annual, fye, lag_months=4)

info_end = acc["available"] - pd.Timedelta(days=1)
lag_check = ((info_end.dt.year - acc["fye"].dt.year) * 12 +
             (info_end.dt.month - acc["fye"].dt.month))
print("Reporting lag in months:", lag_check.value_counts().to_dict())

nondec = (acc["fye"].dt.month != 12).mean()
print(f"Firm-years with a non-December fiscal year end: {100*nondec:.1f}%")
print(f"\nAccounting observations: {len(acc):,} | securities: {acc['symbol'].nunique():,}")

In [ ]:
panel = add_accounting_characteristics(panel, acc)

ACCOUNTING = ["bm", "operatingprofitability", "totalassetgrowth", "accrualschange",
              "roa", "investment", "debttoprice", "salestoprice", "cftoprice",
              "earningstoprice", "grossprofitability"]
for ch in ACCOUNTING:
    print(f"  {ch:24s} {panel[ch].notna().sum():>10,}  ({100*panel[ch].notna().mean():>5.1f}%)")

## Comparison with the source study

Levels are not expected to coincide: here the full universe is used, before the size restriction and the complete-case filter.

In [ ]:
CHARS = MARKET + ["beta", "vola"] + ACCOUNTING

reference = {
    "size": 13.22, "bm": -0.55, "operatingprofitability": 0.06,
    "totalassetgrowth": 0.07, "ret_2_12": 0.09, "issues_1_36": 0.09,
    "accrualschange": -0.04, "roa": 0.05, "investment": 1.02, "ret_1": 0.01,
    "ret_12_36": 0.09, "dy": 0.03, "beta": 0.83, "vola": 0.32, "turnover": 0.02,
    "debttoprice": 0.84, "salestoprice": 2.16, "cftoprice": 0.14,
    "earningstoprice": 0.04, "issues_1_12": 0.02, "fcdispersion": -2.05,
    "grossprofitability": 0.30,
}

cmp = pd.DataFrame({
    "mean": panel[CHARS].mean(),
    "median": panel[CHARS].median(),
    "std": panel[CHARS].std(),
    "coverage%": (100 * panel[CHARS].notna().mean()).round(1),
    "DO_2021_mean": pd.Series(reference),
})
print(cmp.round(3).to_string())

`size` differs by log 1000 because the source expresses market value in thousands. `investment` follows the stated definition, which cannot match the reported mean of 1.02 (Section 3.4.5).

## Observations per month

The number of securities with a complete set of characteristics determines the size
of each cross-section, and is the point of comparison with the 832 firms per month
reported by Drobetz and Otto.

In [ ]:
complete = panel[CHARS + ["target"]].notna().all(axis=1)
per_month_all = panel.dropna(subset=["target"]).groupby(level="date").size()
per_month_cc = panel[complete].groupby(level="date").size()

print(f"Security-months with a target        : {panel['target'].notna().sum():>10,}")
print(f"Security-months complete on all 22   : {int(complete.sum()):>10,}")
print(f"Securities appearing in complete case: "
      f"{panel[complete].index.get_level_values('symbol').nunique():>10,}")
print("\nSecurities per month, selected years:")
print(f"{'year':>6} {'with target':>12} {'complete case':>15}")
for y in [2000, 2005, 2010, 2015, 2020, 2025]:
    a = per_month_all[per_month_all.index.year == y]
    b = per_month_cc[per_month_cc.index.year == y]
    print(f"{y:>6} {a.mean() if len(a) else 0:>12.0f} {b.mean() if len(b) else 0:>15.0f}")
print("\nDrobetz and Otto (2021) report 832 firms per month.")

## Save

In [ ]:
keep = (["ret", "rf", "retx", "target", "MV", "P", "NOSH", "AF", "VO", "DPS"]
        + CHARS + ["fy", "fye", "available"])
keep = [c for c in keep if c in panel.columns]

out_df = panel[keep].reset_index()
out_df.to_parquet(os.path.join(DATA_DIR, "panel_characteristics.parquet"), index=False)

p = os.path.join(DATA_DIR, "panel_characteristics.parquet")
print(f"  panel_characteristics.parquet  {len(out_df):>12,} rows x {out_df.shape[1]} cols  "
      f"({os.path.getsize(p)/1e6:.1f} MB)")

### Price screen and cumulative returns

Standard deviation of the return-based characteristics with and without the price screen (Section 3.4.4).

In [ ]:
# Recompute the return-based characteristics without the price screen,
# to quantify how the screen propagates to the cumulative return windows.
p_ns = build_skeleton(monthly)
p_ns = add_returns(p_ns, rf, apply_reversal_filter=True)
p_ns = add_market_characteristics(p_ns)

print(f"{'characteristic':14s} {'no screen':>12s} {'with screen':>12s}")
for ch in ["ret_1", "ret_2_12", "ret_12_36"]:
    print(f"{ch:14s} {p_ns[ch].std():>12.2f} {panel[ch].std():>12.2f}")

del p_ns

### Alternative windows for ret_12_36

Mean of ret_12_36 under alternative windows (Section 3.5).

In [ ]:
for lag, n, label in [(11, 25, "t-35..t-11  (current)"),
                      (12, 24, "t-35..t-12  (no overlap)"),
                      (12, 25, "t-36..t-12  (literal)"),
                      (12, 23, "t-34..t-12  (GKX-style gap)")]:
    v = _cum_window(panel["retx"], lag, n)
    print(f"{label:26s} mean {v.mean():>7.3f}   (Drobetz-Otto: 0.090)")

### Units of accounting data

Median log book-to-market after dividing accounting items by 1,000.

In [ ]:
bm = pd.read_parquet(os.path.join(DATA_DIR, "panel_characteristics.parquet"), columns=["bm"])["bm"]
m = bm.median()
print(f"median bm {m:.2f} -> raw CE/MV ratio {1000*np.exp(m):.0f}")